# Forest Cover Type Classification

**Tabular Classification Project**

## 2 · Project Overview

Classify **forest cover type** (7 classes) from cartographic variables: elevation, aspect, slope, distances to hydrology/roads/fire points, hillshade, and soil/wilderness binary indicators. Synthetic dataset with ~10,000 rows sampled from the original 580K-row dataset pattern.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given cartographic measurements (elevation, slope, aspect, distances, hillshade, soil type, wilderness area), predict which of 7 forest cover types is present.

## 5 · Why This Project Matters

- **Forest management** and ecological monitoring at scale.
- A challenging **7-class problem** with class imbalance.
- Mostly numeric features with some binary indicator variables.
- Tests model performance on geospatial/environmental data.
- One of the classic ML benchmark datasets.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~10,000 |
| Features | 14 (Elevation, Aspect, Slope, distances, hillshade values, Wilderness_Area, Soil_Type) |
| Target | `Cover_Type` (7 classes: 0-6) |
| Class balance | Imbalanced — types 0,1 dominate |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: UCI Covertype dataset (Blackard & Dean, 1999).
**License**: Public domain / educational.
**Note**: Synthetic reproduction of forest cover patterns from cartographic data.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "Cover_Type"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Forest Cover Type Classification


## 11 · Dataset Download or Loading

In [4]:
np.random.seed(SEED)
n = 10000

elevation = np.random.normal(2900, 300, n).clip(1800, 3900).astype(int)
aspect = np.random.randint(0, 360, n)
slope = np.random.exponential(14, n).clip(0, 60).astype(int)
h_dist_hydro = np.random.exponential(250, n).clip(0, 1400).astype(int)
v_dist_hydro = np.random.normal(40, 60, n).clip(-170, 350).astype(int)
h_dist_road = np.random.exponential(2300, n).clip(0, 7200).astype(int)
hillshade_9am = np.random.normal(212, 28, n).clip(0, 254).astype(int)
hillshade_noon = np.random.normal(223, 20, n).clip(0, 254).astype(int)
hillshade_3pm = np.random.normal(142, 38, n).clip(0, 254).astype(int)
h_dist_fire = np.random.exponential(1800, n).clip(0, 6900).astype(int)
wilderness = np.random.choice([0, 1, 2, 3], n, p=[0.3, 0.3, 0.25, 0.15])
soil = np.random.randint(0, 10, n)

# Cover type based on elevation and wilderness
score = elevation / 500.0 + wilderness * 0.5 + slope * 0.05 + np.random.normal(0, 0.8, n)
cover = pd.cut(score, bins=7, labels=False)

df = pd.DataFrame({
    'Elevation': elevation, 'Aspect': aspect, 'Slope': slope,
    'Horizontal_Distance_To_Hydrology': h_dist_hydro,
    'Vertical_Distance_To_Hydrology': v_dist_hydro,
    'Horizontal_Distance_To_Roadways': h_dist_road,
    'Hillshade_9am': hillshade_9am, 'Hillshade_Noon': hillshade_noon,
    'Hillshade_3pm': hillshade_3pm,
    'Horizontal_Distance_To_Fire_Points': h_dist_fire,
    'Wilderness_Area': wilderness, 'Soil_Type': soil,
    'Cover_Type': cover,
})

# Ensure all 7 classes exist
for cls in range(7):
    if (df['Cover_Type'] == cls).sum() < 10:
        idx = np.random.choice(df.index, 20, replace=False)
        df.loc[idx, 'Cover_Type'] = cls

print(f"Dataset shape: {df.shape}")
print(f"Cover type distribution:\n{df['Cover_Type'].value_counts().sort_index()}")
df.head()

Dataset shape: (10000, 13)
Cover type distribution:
Cover_Type
0      78
1     888
2    3170
3    3750
4    1711
5     368
6      35
Name: count, dtype: int64


,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Wilderness_Area,Soil_Type,Cover_Type
0,3049,119,23,399,60,7161,221,212,94,1724,3,9,4
1,2858,336,19,579,-40,703,223,234,142,4564,2,5,4
2,3094,30,3,60,84,221,234,236,126,537,1,0,3
3,3356,180,27,275,66,1790,250,207,148,811,1,1,5
4,2829,161,5,335,121,469,246,250,122,1968,1,1,2


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (10000, 13)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
Cover_Type
3    3750
2    3170
4    1711
1     888
5     368
0      78
6      35
Name: count, dtype: int64

Dtypes:
Elevation                             int64
Aspect                                int32
Slope                                 int64
Horizontal_Distance_To_Hydrology      int64
Vertical_Distance_To_Hydrology        int64
Horizontal_Distance_To_Roadways       int64
Hillshade_9am                         int64
Hillshade_Noon                        int64
Hillshade_3pm                         int64
Horizontal_Distance_To_Fire_Points    int64
Wilderness_Area                       int64
Soil_Type                             int32
Cover_Type                            int64
dtype: object

Target 'Cover_Type' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology',
            'Hillshade_9am', 'Hillshade_Noon']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3, i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(f"{col} by Cover Type")
plt.suptitle("")
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

corr = df.select_dtypes(include='number').corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True, annot_kws={"size": 7})
ax.set_title("Feature Correlations")
plt.tight_layout()
plt.show()
print(f"Features: {len(df.columns) - 1}")

Features: 12


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df[TARGET].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title("Forest Cover Type Distribution")
axes[0].set_xlabel("Cover Type")
df[TARGET].value_counts().sort_index().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts().sort_index()}")

Class distribution:
Cover_Type
0      78
1     888
2    3170
3    3750
4    1711
5     368
6      35
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().sort_index().to_dict()}")

Train: (8000, 12), Test: (2000, 12)
Train target dist: {0: 63, 1: 710, 2: 2536, 3: 3000, 4: 1369, 5: 294, 6: 28}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
X_train = X_train.copy(); X_test = X_test.copy()
X_train['Elevation_x_Slope'] = X_train['Elevation'] * X_train['Slope']
X_test['Elevation_x_Slope'] = X_test['Elevation'] * X_test['Slope']
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean; X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (13): ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points', 'Wilderness_Area', 'Soil_Type', 'Elevation_x_Slope']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
n_classes = len(np.unique(y_train))
if n_classes == 2:
    y_prob_bl = baseline.predict_proba(X_test)[:, 1]
else:
    y_prob_bl = None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.4675
Precision: 0.4242
Recall   : 0.4675
F1       : 0.4285


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision  Recall  Time Taken
Model                                                                                                        
NearestCentroid                  0.3980           0.412185  0.717968  0.421583   0.461050  0.3980    0.026074
LinearDiscriminantAnalysis       0.5445           0.352057  0.782277  0.535797   0.540914  0.5445    0.030607
GaussianNB                       0.4890           0.342285  0.742400  0.475112   0.489688  0.4890    0.025719
LogisticRegression               0.5580           0.331454  0.792002  0.545424   0.555711  0.5580    0.113693
QuadraticDiscriminantAnalysis    0.4975           0.323448  0.758155  0.491875   0.495592  0.4975    0.027189
LGBMClassifier                   0.5210           0.315750  0.764283  0.509978   0.515113  0.5210    3.581282
CatBoostClassifier               0.5135           0.299829  0.766463  0.502362   0.503

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="macro_f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: lgbm
Accuracy : 0.5360
F1       : 0.5247


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test).ravel()
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.5485  F1: 0.5358


LightGBM  Acc: 0.5140  F1: 0.5028


XGBoost   Acc: 0.5105  F1: 0.4986


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
CatBoost               0.5485  0.5358     0.5418  0.5485
FLAML                  0.5360  0.5247     0.5308  0.5360
LightGBM               0.5140  0.5028     0.5073  0.5140
XGBoost                0.5105  0.4986     0.5049  0.5105
Logistic Regression    0.4675  0.4285     0.4242  0.4675

Top 3: ['CatBoost', 'FLAML', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  CatBoost
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        15
           1       0.57      0.31      0.41       178
           2       0.56      0.63      0.59       634
           3       0.55      0.65      0.60       750
           4       0.49      0.40      0.44       342
           5       0.59      0.23      0.33        74
           6       0.00      0.00      0.00         7

    accuracy                           0.55      2000
   macro avg       0.40      0.32      0.34      2000
weighted avg       0.54      0.55      0.54      2000


  FLAML
              precision    recall  f1-score   support

           0       0.17      0.07      0.10        15
           1       0.54      0.29      0.38       178
           2       0.55      0.62      0.58       634
           3       0.54      0.65      0.59       750
           4       0.51      0.35      0.41       342
           5   

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: CatBoost
Error rate: 0.4515 (903 / 2000)

Errors by true class:
      errors  total  error_rate
true                           
0         15     15    1.000000
1        122    178    0.685393
2        236    634    0.372240
3        260    750    0.346667
4        206    342    0.602339
5         57     74    0.770270
6          7      7    1.000000


## 25 · Interpretation and Business Insight

- **Elevation** is the strongest predictor — different cover types grow at different altitudes.
- **Wilderness area** type captures ecological zone differences.
- **Slope** and **aspect** affect sun exposure and moisture, influencing vegetation.
- Distance features (hydrology, roads, fire) reflect landscape accessibility.
- Hillshade captures light exposure patterns across the day.

## 26 · Limitations

1. Synthetic reproduction — original dataset has 580K rows with 54 features.
2. Reduced feature set (10 soil types instead of 40 binary indicators).
3. Class imbalance — some cover types are rare.
4. No temporal data (seasonal variation, fire history).
5. Cartographic data may not capture microhabitat variation.

## 27 · How to Improve This Project

1. Use the full 580K-row dataset with all 54 features.
2. Add satellite imagery features (NDVI, spectral bands).
3. Include temporal fire history and climate data.
4. Engineer elevation bands and aspect quadrants.
5. Use spatial cross-validation to avoid geographic leakage.

## 28 · Production Considerations

- Remote sensing and GIS integration for forest monitoring.
- Wildfire risk assessment by cover type.
- Timber management and conservation planning.
- Regular retraining with updated satellite data.

## 29 · Common Mistakes

1. Not recognizing elevation as the dominant feature.
2. Using accuracy on imbalanced 7-class data.
3. Ignoring spatial autocorrelation in train/test splits.
4. Not trying ordinal models (cover types may be ordered by elevation).
5. Over-fitting on rare cover types without proper handling.

## 30 · Mini Challenge / Exercises

1. Build a model with only Elevation and Wilderness_Area — what accuracy?
2. Which cover type is hardest to predict? Why?
3. Create elevation bands (low/mid/high) as features.
4. Compare weighted F1 vs macro F1 — which cover types lose?

## 31 · Final Summary / Key Takeaways

1. Forest cover type is a classic 7-class geospatial classification.
2. Elevation is the dominant predictor of vegetation type.
3. Class imbalance requires careful metric selection.
4. Full dataset (580K rows) would significantly improve models.
5. Real applications combine cartographic data with satellite imagery.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.5485,
    "f1": 0.5358,
    "precision": 0.5418,
    "recall": 0.5485
  },
  "LightGBM": {
    "accuracy": 0.514,
    "f1": 0.5028,
    "precision": 0.5073,
    "recall": 0.514
  },
  "XGBoost": {
    "accuracy": 0.5105,
    "f1": 0.4986,
    "precision": 0.5049,
    "recall": 0.5105
  },
  "Logistic Regression": {
    "accuracy": 0.4675,
    "f1": 0.4285,
    "precision": 0.4242,
    "recall": 0.4675
  },
  "FLAML": {
    "accuracy": 0.536,
    "f1": 0.5247,
    "precision": 0.5308,
    "recall": 0.536
  }
}
